## Code to preprocess Chenchen Zhu's datafrom the large intestine using 3D model, 2D layers, and RNA molecules

## Install and import libraries

In [20]:
%pip install requests pandas ipywidgets hra_jupyter_widgets

import os
import requests
import pandas as pd
import gzip
from pprint import pprint


Note: you may need to restart the kernel to use updated packages.


In [21]:
import ipywidgets as widgets

# Import hra-jupyter-widgets. For documentation, please see https://github.com/x-atlas-consortia/hra-jupyter-widgets/blob/main/usage.ipynb
from hra_jupyter_widgets import (
    BodyUi,               # CCF 3D Body UI
    CdeVisualization,     # Cell Distance Explorer (CDE)
    Eui,                  # Exploration User Interface (EUI)
    EuiOrganInformation,  # Organ Information
    FtuExplorer,          # Functional Tissue Unit (FTU) Explorer
    FtuExplorerSmall,     # FTU Explorer (small version)
    MedicalIllustration,  # Medical Illustration Viewer (2D FTU Viewer)
    ModelViewer,          # GLB model viewer
    NodeDistVis,          # VCCF 3D Node Distance Visualization
    Rui,                  # Registration User Interface (RUI)
)

## Download and save layers locally, then as DataFrames

In [22]:

# Make sure the data folder is present
folder = "data"

if not os.path.exists(folder):
    os.makedirs(folder)
    print(f"Folder '{folder}' created.")
else:
    print(f"Folder '{folder}' already exists.")

Folder 'data' already exists.


In [23]:
# load data from HRA CDN
def get_layer(layer_number:int):
  """Downloads the corresponding layer nodes CSV from the HRA CDN

  Args:
      layer_number (int): Number of layer
      
  Returns:
      DataFrame
  """

  url = f'https://cdn.humanatlas.io/image-store/vccf-data-cell-nodes/unpublished/colon-xenium-stanford/layer_{layer_number}-nodes.csv'
  
  dtype = {'x': 'float', 'y': 'float', 'Cell Type': 'string'}
  

  
  file = f'layer_{layer_number}.csv'
  file_path = f'{folder}/{file}'
    
      # Check if the file exists
  if not os.path.exists(file_path):
    response = requests.get(url)
    if response.status_code == 200:
      print(f'Found {url}, downloading now.')
      
      # If the file doesn't exist, download it as a df
      df = pd.read_csv(url, dtype=dtype)
      df.to_csv(f'{file_path}', index=False)
      print(f"File downloaded and saved at {file_path}.")
      return df
    
    else:
      print('URL not found.')
  
  else:
    df = pd.read_csv(file_path, dtype=dtype)
    print(f"File already exists at {file_path}, loading from storage.")
    return df

## Load and merge all layers, then add z-axis

In [ ]:
# define z-offset
offset = 100

# set scale factor
scale = 10

# create emptt df to capture all layers
df_concat = pd.DataFrame()

for i in range(0,31): # adjust range as needed
  result = get_layer(i)
  
  if result is not None:
    
    # scale up
    result[['x', 'y']] = result[['x', 'y']].apply(lambda n: n * scale)
    
    # add z-axis to layer
    result['z'] = offset * i
    
    df_concat = pd.concat([df_concat, result], ignore_index=True)

df_concat

URL not found.
URL not found.
URL not found.
File already exists at data/layer_3.csv, loading from storage.
File already exists at data/layer_4.csv, loading from storage.
File already exists at data/layer_5.csv, loading from storage.
File already exists at data/layer_6.csv, loading from storage.
File already exists at data/layer_7.csv, loading from storage.
File already exists at data/layer_8.csv, loading from storage.
File already exists at data/layer_9.csv, loading from storage.
File already exists at data/layer_10.csv, loading from storage.
File already exists at data/layer_11.csv, loading from storage.
File already exists at data/layer_12.csv, loading from storage.
File already exists at data/layer_13.csv, loading from storage.
File already exists at data/layer_14.csv, loading from storage.
File already exists at data/layer_15.csv, loading from storage.
File already exists at data/layer_16.csv, loading from storage.
File already exists at data/layer_17.csv, loading from storage.
Fi

,x,y,Cell Type,z
0,10779.600000,25568.200000,Immature Goblet,300
1,10790.100000,25342.300000,Tuft,300
2,10826.500000,25540.900000,TA1,300
3,10918.900000,25473.800000,Immature Goblet,300
4,12534.600000,25546.400000,CD4+,300
...,...,...,...,...
2551915,10635.754628,12392.438947,Unknown_lowcount,3000
2551916,10413.392509,12482.936693,Best4+ Enterocytes,3000
2551917,9748.574498,13189.830025,Best4+ Enterocytes,3000
2551918,9777.576344,13143.309918,TA2,3000


## Get extents of bounding box

In [25]:
width = df_concat['x'].max()
depth = df_concat['z'].max()
height = df_concat['y'].max()

print(f'Bounding box (WxDxH): {width}x{depth}x{height}')

Bounding box (WxDxH): 40314.41703690463x3000x40382.87701260661


## Visualize with `node-dist-vis`

In [26]:
# Next, let's define a function that turns a DataFrame into a node list that can then be passed into the CdeVisualization or NodeDistVis widget
def make_node_list(df: pd.DataFrame, category:str):
  """Turn a DataFrame into a list of dicts for passing them into a HRA widget

  Args:
      df (pd.DataFrame): A DataFrame with cells
  """

  node_list = [{'x': row['x'], 'y': row['y'], 'z': row['z'], 'Cell Type': row[f'{category}']}
               for index, row in df.iterrows()]

  return node_list

In [27]:
node_list = make_node_list(df_concat, 'Cell Type')
edge_list = [["Cell ID", "Target ID", "X1", "Y1", "Z1", "X2", "Y2", "Z2"]] # this makes an empty edges list

In [28]:
node_dist_vis = NodeDistVis(
    nodes=node_list,
    node_target_key="Cell Type",
    edges=edge_list
)
display(node_dist_vis)

NodeDistVis(color_map=None, color_map_data=None, color_map_key=None, color_map_keys=None, edge_keys=None, edge…

## Open, pre-process, visualize RNA molecules

In [29]:
dict_df_transcripts = {}

for i in range(4, 5):
  try_path = f'{folder}/{i}.csv.gz'
  if os.path.exists(try_path):
    df_new = pd.read_csv(try_path, compression='gzip')
    dict_df_transcripts[f'layer_{i}'] = df_new
  else:
    continue
  

dict_df_transcripts

{'layer_4':                   x           y          z feature_name
 0        1124.42590   184.81421  15.259870         SDC1
 1        1124.50160   184.44824  14.978872        BANK1
 2        1100.40500   170.34253  14.892603       RETNLB
 3        1022.40063   177.99756  15.489591      TNFAIP3
 4        1096.56180   171.15466  14.931789        CLCA1
 ...             ...         ...        ...          ...
 6647721  2438.57700  3822.54930  11.680466        ITGAM
 6647722  2466.50400  3826.75930  11.857481      SULT1B1
 6647723  2439.91000  3822.73950  12.122388       IL18R1
 6647724  2456.00600  3825.03610  11.687558      C2orf88
 6647725  2438.57700  3822.54930  11.889899         MUC2
 
 [6647726 rows x 4 columns]}

In [30]:
# merge dfs for vis
pprint(dict_df_transcripts.keys())

dict_keys(['layer_4'])


In [34]:
df_vis = pd.concat(list(dict_df_transcripts.values())[:3], ignore_index=True).head(400000)
df_vis

,x,y,z,feature_name
0,1124.42590,184.81421,15.259870,SDC1
1,1124.50160,184.44824,14.978872,BANK1
2,1100.40500,170.34253,14.892603,RETNLB
3,1022.40063,177.99756,15.489591,TNFAIP3
4,1096.56180,171.15466,14.931789,CLCA1
...,...,...,...,...
399995,916.58080,1134.66780,19.160494,LYZ
399996,915.29090,1135.58100,18.494787,LYZ
399997,912.02720,1137.92650,17.636984,LYZ
399998,910.54860,1138.96780,19.163200,NegControlCodeword_0532


In [35]:
node_list = make_node_list(df_vis, 'feature_name')

In [36]:
node_dist_vis = NodeDistVis(
    nodes=node_list,
    node_target_key="Cell Type",
    edges=edge_list
)
display(node_dist_vis)

NodeDistVis(color_map=None, color_map_data=None, color_map_key=None, color_map_keys=None, edge_keys=None, edge…